In [6]:
import glob
import pandas as pd
import re
files = glob.glob(f'sim_stats/*/*/*/reward.csv')

files = [
    f for f in files
    if 1 <= int(re.search(r'\\SEED(\d+)\\', f).group(1)) <= 3
]
dfs = [pd.read_csv(f) for f in files]
agents = [tuple(f.split('\\')[1:4]) for f in files]
df = pd.concat(dfs, keys=agents, names=['policy', 'scenario', 'seed'])
# df0 = df = df.droplevel(-1).groupby(['policy', 'scenario', 'seed'], sort=False).mean()
# print(df)
# reward_sum = df.groupby(['policy', 'scenario'])['reward'].sum()
reward_sum = (
    df.groupby(['policy', 'scenario', 'seed'])['reward']
      .sum()
      .groupby(['policy', 'scenario'])
      .mean()
)
# qos_mean = df.groupby(['policy', 'scenario'])['qos_reward'].mean()
pc_mean = df.groupby(['policy', 'scenario'])['pc_kw'].mean()
# dp_mean = df.groupby(['policy', 'scenario'])['drop_ratio'].mean()
# dp_mean = df.groupby(['policy', 'scenario', 'seed'])['drop_ratio'].mean().loc[(slice(None), 'B', 'SEED1')]
print(reward_sum)
# print(qos_mean)
print(pc_mean)
# print(dp_mean)

policy                                      scenario
A_w_str=150.0_w_sen=1.5_w_tol=1.5_w_pc=1.0  A          -194.715728
Name: reward, dtype: float64
policy                                      scenario
A_w_str=150.0_w_sen=1.5_w_tol=1.5_w_pc=1.0  A           0.323472
Name: pc_kw, dtype: float64


In [61]:
import pickle
import numpy as np
from collections import Counter

with open('trajectory/ab.pkl', 'rb') as f:
    data = pickle.load(f)

# all_actions = np.concatenate([traj['actions'] for traj in data], axis=0)
# all_actions = all_actions.squeeze()

# # 如果是单动作（离散动作），直接统计
# counts = Counter(all_actions)
# print("Action distribution:", counts)
# print("动作标准差:", np.std(all_actions))

# import matplotlib.pyplot as plt

# unique, freq = np.unique(all_actions, return_counts=True)
# plt.bar(unique, freq / freq.sum())
# plt.title("Action Distribution")
# plt.xlabel("Action Value")
# plt.ylabel("Frequency")
# plt.show()

# data = data[:-120]

# # 保存回原文件（或另存为新文件）
# with open('trajectory/1.pkl', 'wb') as f:
#     pickle.dump(data, f)

# for traj in data:
#     traj['observations'][:, 0, 0] *= 10   # 每行的第一个数乘以10

print(len(data))
# for item in data:
#     item['observations'] = np.delete(item['observations'], 0, axis=2)

# with open('trajectory/1.pkl', 'wb') as f:
#     pickle.dump(data, f)
# print(len(data[0]['actions']))
# for d in data:
    # print(d['observations'].shape)
    # print(d['rewards'])
    # rewards = d['rewards']
    # rtg = np.flip(np.cumsum(np.flip(rewards, axis=0), axis=0), axis=0)
    # print(rtg[0])
    # print(d['actions'])

150


In [47]:
import pickle
from pathlib import Path

folder = Path(r"trajectory")
target = folder / "ab.pkl"
source = folder / "c.pkl"

with target.open("rb") as f:
    ab = pickle.load(f)

with source.open("rb") as f:
    c = pickle.load(f)

if not isinstance(ab, list) or not isinstance(c, list):
    raise TypeError("两个文件的内容都必须是轨迹列表")

n_ab, n_c = len(ab), len(c)
ab.extend(c)

# 先写临时文件，成功后再替换原文件
temp = target.with_suffix(".tmp")
with temp.open("wb") as f:
    pickle.dump(ab, f, protocol=pickle.HIGHEST_PROTOCOL)

temp.replace(target)
print(f"合并完成：{n_ab} + {n_c} = {len(ab)} 条轨迹")

合并完成：50 + 100 = 150 条轨迹
